# Multi-Agent Healthcare System with Episodic Memory

# Multi-Agent Healthcare System with Episodic Memory

## Introduction

This notebook demonstrates how to implement a **multi-agent healthcare system with episodic memory** using the AgentCore Memory SDK and Strands memory hooks. This approach provides automatic memory management without manual API calls.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:----------------------------------------------------------------------------------|
| Tutorial type       | Episodic Memory with Multi-Agent Coordination                                    |
| Agent type          | Healthcare Assistant System                                                      |
| Agentic Framework   | Strands Agents with Memory Hooks                                                 |
| LLM model           | Anthropic Claude Sonnet 4                                                        |
| Tutorial components | Episodic Memory, Memory Hooks, HealthLake Integration                           |
| Example complexity  | Intermediate                                                                     |

You will learn:

- How to use the MemoryClient SDK for episodic memory
- Creating memory hooks for automatic memory management
- Implementing specialized agents with shared episodic memory
- Integrating real-time HealthLake FHIR queries

## Scenario Context

We'll create a **Healthcare Assistant System** with:
1. A **Supervisor Agent** that routes patient questions
2. A **Claims Agent** for insurance and billing
3. A **Demographics Agent** for patient information
4. A **Medication Agent** for prescriptions

All agents use memory hooks to automatically save conversations to episodic memory.

## Architecture
<div style="text-align:left">
    <img src="architecture.png" width="75%" />
</div>

## Prerequisites

- Python 3.9+
- AWS credentials with Bedrock and AgentCore Memory permissions
- Amazon HealthLake datastore (optional)

Let's get started!

## Step 1: Environment Setup
Let's begin by installing all the necessary libraries for this tutorial.

In [ ]:
%pip install -qr ./requirements.txt

## Step 2: Create IAM Role for Memory Execution

Custom memory strategies require an IAM role with permissions to invoke Bedrock models.

In [ ]:
import logging
import json
from datetime import datetime
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare-assistant")

from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

In [ ]:
MEMORY_NAME = "healthcare_episodic_memory"
PATIENT_ID = "b2055b4d-ac17-4d94-8c5b-3395e4c334dd"
REGION = "us-east-1"
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Configuration:")
print(f"  Memory Name: {MEMORY_NAME}")
print(f"  Patient ID: {PATIENT_ID}")
print(f"  Region: {REGION}")
print(f"  Session ID: {SESSION_ID}")

## Step 3: Create Memory with Episodic Strategy

In [ ]:
client = MemoryClient(region_name=REGION)

strategies = [
    {
        StrategyType.EPISODIC.value: {
            "name": "HealthcareEpisodes",
            "description": "Captures healthcare interactions as episodes",
            "namespaces": ["healthcare/{actorId}/{sessionId}"],
            "extraction": {"appendToPrompt": "Extract patient interactions with healthcare agents"},
            "consolidation": {"appendToPrompt": "Consolidate healthcare conversations"},
            "reflection": {"appendToPrompt": "Generate insights from patient care patterns", "namespaces": ["healthcare/{actorId}"]}
        }
    }
]

try:
    memory = client.create_memory_and_wait(name=MEMORY_NAME, strategies=strategies, description="Healthcare system with episodic memory", event_expiry_days=90)
    memory_id = memory['id']
    logger.info(f"Created memory: {memory_id}")
except ClientError as e:
    if "already exists" in str(e):
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(MEMORY_NAME)), None)
        logger.info(f"Using existing memory: {memory_id}")
    else:
        raise

print(f"Memory ID: {memory_id}")

## Step 4: Configure HealthLake Integration

In [ ]:
client = MemoryClient(region_name=REGION)

# Note: Once episodic is GA, this will work with the SDK
# For now, this shows the intended API
try:
    memory = client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Healthcare system with episodic memory",
        strategies=[{
            StrategyType.EPISODIC.value: {  # Will be available in GA SDK
                "extraction": {
                    "appendToPrompt": "Extract patient interactions with healthcare agents"
                },
                "consolidation": {
                    "appendToPrompt": "Consolidate healthcare conversations"
                },
                "reflection": {
                    "appendToPrompt": "Generate insights from patient care patterns",
                    "namespaces": ["healthcare/{actorId}"]
                }
            }
        }]
    )
    
    memory_id = memory.memory_id
    print(f"✅ Memory created: {memory_id}")
    
except Exception as e:
    print(f"⚠️ Episodic strategy not yet available in GA SDK")
    print(f"   Use multi-agent-healthcare-memory.ipynb for preview access")
    raise

## Step 5: Create Memory Hook Provider

In [ ]:
class HealthcareMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
    
    def save_memory(self, event: AfterInvocationEvent):
        try:
            messages = event.agent.messages
            if len(messages) >= 2:
                user_msg = None
                assistant_msg = None
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg:
                        user_msg = msg["content"][0]["text"]
                        break
                if user_msg and assistant_msg:
                    self.client.create_event(memory_id=self.memory_id, actor_id=self.actor_id, session_id=self.session_id, messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")])
                    logger.info("Memory saved")
        except Exception as e:
            logger.error(f"Error saving memory: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterInvocationEvent, self.save_memory)

print("Memory hook provider defined")

## Step 6: Create HealthLake Query Tools

In [ ]:
class HealthcareMemoryHooks(HookProvider):
    """Automatic memory management for healthcare agents"""
    
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, branch_name: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.branch_name = branch_name
    
    def save_memory(self, event: AfterInvocationEvent):
        """Save conversation to memory after agent invocation"""
        try:
            # Extract conversation from event
            user_message = event.input_data
            assistant_message = str(event.result)
            
            # Save to memory with branch
            self.client.create_event(
                memory_id=self.memory_id,
                actor_id=self.actor_id,
                payload=[
                    {"conversational": {"content": {"text": user_message}, "role": "USER"}},
                    {"conversational": {"content": {"text": assistant_message}, "role": "ASSISTANT"}}
                ],
                branch={"name": self.branch_name}  # Branch isolation
            )
            
            logger.info(f"Memory saved to branch: {self.branch_name}")
        except Exception as e:
            logger.error(f"Error saving memory: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterInvocationEvent, self.save_memory)
        logger.info(f"Memory hooks registered for {self.branch_name}")

print("✅ Memory hook provider defined")

## Step 7: Create Agents with Memory Hooks

In [ ]:
supervisor_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, SESSION_ID)
claims_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, SESSION_ID)
demographics_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, SESSION_ID)
medication_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, SESSION_ID)

supervisor = Agent(model="global.anthropic.claude-sonnet-4-20250514-v1:0", system_prompt="Route patient questions to specialized agents.", hooks=[supervisor_hooks])
claims_agent = Agent(model="global.anthropic.claude-sonnet-4-20250514-v1:0", system_prompt="Handle insurance claims using get_patient_claims tool.", tools=[get_patient_claims], hooks=[claims_hooks])
demographics_agent = Agent(model="global.anthropic.claude-sonnet-4-20250514-v1:0", system_prompt="Handle patient demographics using get_patient_demographics tool.", tools=[get_patient_demographics], hooks=[demographics_hooks])
medication_agent = Agent(model="global.anthropic.claude-sonnet-4-20250514-v1:0", system_prompt="Handle medications using get_patient_medications tool.", tools=[get_patient_medications], hooks=[medication_hooks])

print("Agents created with automatic memory hooks")

## Step 8: Interactive Chat

In [ ]:
# Create memory hooks for each agent
supervisor_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, "main")
claims_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, "claims_agent")
demographics_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, "demographics_agent")
medication_hooks = HealthcareMemoryHooks(memory_id, client, PATIENT_ID, "medication_agent")

# Create agents with hooks
supervisor = Agent(
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="Route patient questions to specialized agents.",
    hooks=[supervisor_hooks]
)

claims_agent = Agent(
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="Handle insurance claims using get_patient_claims tool.",
    tools=[get_patient_claims],
    hooks=[claims_hooks]
)

demographics_agent = Agent(
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="Handle patient demographics using get_patient_demographics tool.",
    tools=[get_patient_demographics],
    hooks=[demographics_hooks]
)

medication_agent = Agent(
    model="global.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="Handle medications using get_patient_medications tool.",
    tools=[get_patient_medications],
    hooks=[medication_hooks]
)

print("✅ Agents created with automatic memory hooks")

## Step 9: Verify Memory Storage

In [ ]:
print(f"Checking memories for session: {SESSION_ID}")

try:
    recent_turns = client.get_last_k_turns(memory_id=memory_id, actor_id=PATIENT_ID, session_id=SESSION_ID, k=5)
    print(f"Total conversation turns: {len(recent_turns)}")
    for i, turn in enumerate(recent_turns, 1):
        print(f"Turn {i}:")
        for message in turn:
            role = message['role']
            content = message['content']['text'][:100] + "..." if len(message['content']['text']) > 100 else message['content']['text']
            print(f"  {role}: {content}")
        print()
except Exception as e:
    print(f"Error checking memories: {e}")
    import traceback
    traceback.print_exc()

## Step 10: Check Long-Term Memory Processing

Episodes and reflections are processed asynchronously.

In [ ]:
print("CHECKING LONG-TERM MEMORY (EPISODES & REFLECTIONS)")
episode_namespace = f"healthcare/{PATIENT_ID}/{SESSION_ID}"
reflection_namespace = f"healthcare/{PATIENT_ID}"
print(f"Episode namespace: {episode_namespace}")
print(f"Reflection namespace: {reflection_namespace}")

try:
    print("\nEPISODES (Session-specific learning)")
    episodes = client.retrieve_memories(memory_id=memory_id, namespace=episode_namespace, query="patient interactions", top_k=10)
    print(f"Found {len(episodes)} episode(s)")
    for i, episode in enumerate(episodes, 1):
        print(f"Episode {i}")
        content = episode.get('content', {})
        if isinstance(content, dict):
            text = content.get('text', '')
            print(f"Content: {text[:200]}..." if len(text) > 200 else f"Content: {text}")
    if not episodes:
        print("No episodes found yet. Episodic processing happens asynchronously.")
except Exception as e:
    print(f"Error retrieving episodes: {e}")

try:
    print("\nREFLECTIONS (Patient-level insights)")
    reflections = client.retrieve_memories(memory_id=memory_id, namespace=reflection_namespace, query="patient care patterns", top_k=10)
    print(f"Found {len(reflections)} reflection(s)")
    for i, reflection in enumerate(reflections, 1):
        print(f"Reflection {i}")
        content = reflection.get('content', {})
        if isinstance(content, dict):
            text = content.get('text', '')
            print(f"Content: {text[:200]}..." if len(text) > 200 else f"Content: {text}")
    if not reflections:
        print("No reflections found yet.")
except Exception as e:
    print(f"Error retrieving reflections: {e}")

print("\nTIP: Use the memory browser for interactive visualization")

## Step 11: Check Long-Term Memory Processing

Episodes and reflections are processed asynchronously. Use the memory browser to view them once ready.

In [ ]:
# Check for episodes and reflections (may take time to process)
# Display Episodes and Reflections
import json
from datetime import datetime

print("=" * 80)
print("CHECKING LONG-TERM MEMORY (EPISODES & REFLECTIONS)")
print("=" * 80)

episode_namespace = f"healthcare/{PATIENT_ID}/{SESSION_ID}"
reflection_namespace = f"healthcare/{PATIENT_ID}"
print(f"Episode namespace: {episode_namespace}")
print(f"Reflection namespace: {reflection_namespace}")

try:
    print(f"\n{'='*80}")
    print("EPISODES (Session-specific learning)")
    print(f"{'='*80}")
    
    episodes_response = client.list_memory_records(
        memoryId=MEMORY_ID,
        namespace=episode_namespace,
        maxResults=50
    )
    
    
    episodes = episodes_response.get('memoryRecordSummaries', [])
    print(f"\n✅ Found {len(episodes)} episode(s)\n")
    print(episodes_response)
    
    for i, episode in enumerate(episodes, 1):
        print(f"\n{'─'*80}")
        print(f"Episode {i}")
        print(f"{'─'*80}")
        print(f"Record ID: {episode.get('recordId', 'N/A')}")
        print(f"Created: {episode.get('createdAt', 'N/A')}")
        
        # Parse content
        content = episode.get('content', {})
        if isinstance(content, str):
            try:
                content = json.loads(content)
            except:
                pass
        
        # Display episode details
        if isinstance(content, dict):
            print(f"\n📝 Episode Content:")
            
            # Show turns if available
            if 'turns' in content:
                turns = content['turns']
                print(f"   Turns: {len(turns)}")
                for j, turn in enumerate(turns[:3], 1):  # Show first 3 turns
                    print(f"\n   Turn {j}:")
                    if 'situation' in turn:
                        print(f"      Situation: {turn['situation'][:100]}...")
                    if 'intent' in turn:
                        print(f"      Intent: {turn['intent'][:100]}...")
            
            # Show embedded reflection if available
            if 'reflection' in content:
                refl = content['reflection']
                print(f"\n   💡 Embedded Reflection:")
                if 'title' in refl:
                    print(f"      Title: {refl['title']}")
                if 'summary' in refl:
                    print(f"      Summary: {refl['summary'][:200]}...")
        else:
            print(f"\nContent: {str(content)[:500]}...")
    
    if not episodes:
        print("⏳ No episodes found yet. Episodic processing happens asynchronously.")
        print("   Episodes may take a few minutes to appear after conversation.")
        
except Exception as e:
    print(f"Error retrieving episodes: {e}")

try:
    print(f"\n\n{'='*80}")
    print("REFLECTIONS (Patient-level insights across all sessions)")
    print(f"{'='*80}")
    
    reflections_response = client.retrieve_memory_records(
        memoryId=MEMORY_ID,
        namespace=reflection_namespace,
        searchCriteria={
            "searchQuery": "(especially holiday wishes)",
            "metadataFilters": [
                {
                    "left": {"metadataKey": "x-amz-agentcore-memory-recordType"},
                    "operator": "EQUALS_TO",
                    "right": {"metadataValue": {"stringValue": "REFLECTION"}}
                }
            ],
            "topK": 10
        },
        maxResults=20
    )
    
    reflections = reflections_response.get('memoryRecordSummaries', [])
    print(f"\n✅ Found {len(reflections)} reflection(s)\n")
    
    for i, reflection in enumerate(reflections, 1):
        print(f"Reflection {i}")
        print(f"{'─'*80}")
        print(f"Record ID: {reflection.get('memoryRecordId', 'N/A')}")
        print(f"Created: {reflection.get('createdAt', 'N/A')}")
        
        content = reflection.get('content', {})
        if isinstance(content, dict) and 'text' in content:
            content_text = content['text']
            try:
                content = json.loads(content_text)
            except:
                content = content_text
        
        if isinstance(content, dict):
            print(f"\n💡 Reflection Content:")
            if 'title' in content:
                print(f"   Title: {content['title']}")
            if 'use_cases' in content:
                print(f"   Use Cases: {content['use_cases']}")
            if 'hints' in content:
                print(f"   Hints: {content['hints']}")
            if 'confidence' in content:
                print(f"   Confidence: {content['confidence']}")
        else:
            print(f"\nContent: {str(content)[:500]}...")
    
    if not reflections:
        print("No reflections found yet.")
except Exception as e:
    print(f"Error retrieving reflections: {e}")

print("\nTIP: Use the memory browser for interactive visualization")

## Summary

### What We Built:
1. **Supervisor Agent** - Orchestrates on main branch
2. **Claims Agent** - Handles insurance on claims_agent branch
3. **Demographics Agent** - Manages patient info on demographics_agent branch
4. **Medication Agent** - Handles medications on medication_agent branch

### Memory Architecture:
- **Short-term**: Each agent has isolated branch
- **Episodes**: Stored per session `healthcare/{actorId}/{sessionId}`
- **Reflections**: Shared across all sessions `healthcare/{actorId}`

### Benefits:
- ✅ Agents don't interfere with each other's conversations
- ✅ All agents contribute to same session's long-term memory
- ✅ Learned patterns (reflections) shared across all patient sessions
- ✅ Complete conversation history maintained per agent

## Cleanup (Optional)

Run this cell to delete the memory and IAM role created in this tutorial.

In [ ]:
# import boto3

#print("Cleanup Options:")
#delete_memory = input("Delete memory? (yes/no): ").strip().lower()
#if delete_memory == 'yes':
#   try:
#       print(f"Deleting memory: {memory_id}")
#       client.delete_memory_and_wait(memory_id=memory_id)
#       print("Memory deleted")
#   except Exception as e:
#       print(f"Error deleting memory: {e}")
# else:
#   print(f"Memory preserved: {memory_id}")

# delete_healthlake = input("Delete HealthLake datastore? (yes/no): ").strip().lower()
# if delete_healthlake == 'yes':
#     try:
#         print(f"Deleting HealthLake datastore: {DATASTORE_ID}")
#         healthlake_client.delete_fhir_datastore(DatastoreId=DATASTORE_ID)
#         print("HealthLake datastore deletion initiated")
#     except Exception as e:
#         print(f"Error deleting HealthLake datastore: {e}")
# else:
#     print(f"HealthLake datastore preserved: {DATASTORE_ID}")

# print("Cleanup complete")